# 04 国家間ネットワーク

国家をノード、国家ペアの関係をエッジとする**無向グラフ**。隣接リスト・隣接行列、**DFS（スタック）**と **BFS（キュー）**、連結成分、協調エッジ（RQ4）。
計算：`network_model.py`

> ネットワーク上の「つながり」は関係があることを表すだけで、政治的な協調関係そのものではありません。協調はエッジの状態で表します。

> **補遺トピック G2・教育用の抽象的な仮想モデル**です。国家（Country A〜E）・利得行列・状態遷移確率・係数はすべて架空で、現実の国際関係・紛争や特定国家の行動を予測するものではありません。扱うのは協調・対話・信頼形成・緊張緩和だけです。

## 準備：実験環境（最初に必ず実行）

G2 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G2 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G2 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G2 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                         # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g2-conflict-cooperation-simulation"    # リポジトリの中の G2 フォルダ（ルートが G2 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g2_root(p):
    return (Path(p) / "game_theory.py").exists() and (Path(p) / "network_model.py").exists()


def find_g2_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G2 のルート
        if is_g2_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g2_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G2 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g2_upload")
    for hit in glob.glob("/content/g2_upload/**/game_theory.py", recursive=True):
        if is_g2_root(Path(hit).parent):
            return Path(hit).parent
    return None


G2_ROOT = find_g2_root()
if G2_ROOT is None:
    raise RuntimeError("G2 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G2_ROOT)
if str(G2_ROOT) not in sys.path:
    sys.path.insert(0, str(G2_ROOT))
if not (G2_ROOT / "textbook" / "fixed_stack.py").exists():
    print("⚠️ textbook/ がありません。network_model.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import agents as ag
import game_theory as gt
import trust_model as tm
import state_transitions as st
import simulation as sim
import network_model as nm
import cooperation_policies as cp
import evaluation as ev
import conflict_charts as cc

cc.setup_style()                        # 文字を大きくするなら cc.setup_style(1.3)
COUNTRIES = ag.load_countries()         # 架空の国家 A〜E
PARAMS = sim.load_parameters()          # シミュレーションのパラメータ（すべて架空）
M = gt.load_payoff_matrix()             # 利得行列（架空）
IDS = [c["id"] for c in COUNTRIES]
RUNS = PARAMS["monte_carlo_runs"]       # モンテカルロの反復回数
YEARS = PARAMS["simulation_years"]
LABELS = {s: gt.label(s) for s in ag.STRATEGIES}

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", cc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


SUMMARY_COLUMNS = [("name", "condition", "{}"), ("cooperation_rate", "cooperation", "{:.3f}"),
                   ("cooperation_rate_sd", "sd", "{:.3f}"), ("mutual_cooperation_rate", "mutual coop.", "{:.3f}"),
                   ("cooperation_state_rate", "Cooperation state", "{:.3f}"), ("tension_rate", "Tension", "{:.3f}"),
                   ("conflict_rate", "Conflict", "{:.3f}"), ("conflict_rate_sd", "sd", "{:.3f}")]

print("G2_ROOT:", G2_ROOT)
print("架空の国家:", ", ".join(f"{c['id']}({LABELS[c['strategy']]})" for c in COUNTRIES))
print(f"モンテカルロ {RUNS} 回 × {YEARS} 年、乱数シード {PARAMS['random_seed']}")

## 1. 4 種類のネットワーク

In [ ]:
#@title Line・Ring・Star・Random
NETS = {kind: nm.make_network(kind, IDS, PARAMS["network"]["random_edge_probability"], PARAMS["network"]["random_seed"])
        for kind in nm.NETWORK_TYPES}
show(cc.plot_networks([(f"{k.capitalize()} network", IDS, e) for k, e in NETS.items()],
                      "Network structures (5 fictional countries)"), "04_networks")
table([{"name": k, **nm.network_metrics(IDS, e)} for k, e in NETS.items()],
      [("name", "network", "{}"), ("nodes", "nodes", "{}"), ("edges", "edges", "{}"),
       ("average_degree", "avg degree", "{:.1f}"), ("components", "components", "{}"),
       ("largest_component", "largest", "{}")])

## 2. 隣接リストと隣接行列

| | 隣接リスト | 隣接行列 |
| --- | --- | --- |
| メモリ | $O(V + E)$ | $O(V^2)$ |
| 隣のノードの列挙 | $O(\text{次数})$ | $O(V)$ |
| 辺の有無の確認 | $O(\text{次数})$ | $O(1)$ |

In [ ]:
E = NETS["ring"]
print("隣接リスト:", nm.adjacency_list(IDS, E))
order, mat = nm.adjacency_matrix(IDS, E)
print("隣接行列:")
for v, row in zip(order, mat):
    print(" ", v, row)
print("格納する値の数の概算:", nm.memory_estimate(IDS, E))

## 3. DFS と BFS（Comparison A）

- DFS：教科書の **FixedStack**（スタック）。再帰版もある
- BFS：教科書の **FixedQueue**（キュー）。無向・重みなしのグラフで**辺の数が最小の経路**を求める
- どちらも $O(V + E)$。到達できるノードは同じだが、訪問順・経路・必要な記憶域（スタック／キューの最大の長さ）が違う

In [ ]:
#@title DFS と BFS の訪問順
adj = nm.adjacency_list(IDS, E)
d, b = nm.dfs(adj, "A"), nm.bfs(adj, "A")
show(cc.plot_search_orders(adj, "A", d, b), "04_search_orders")
print("DFS 経路 A → D:", nm.path_to(d["parent"], "D"), " BFS 経路 A → D:", nm.path_to(b["parent"], "D"))

In [ ]:
import time
rows, sizes = [], [100, 1_000, 10_000, 50_000]
times = {"DFS (stack)": [], "BFS (queue)": []}
for n in sizes:                                   # TODO：ノード数・ネットワークの種類を変える
    nodes = nm.node_names(n)
    edges = nm.make_network("ring", nodes)
    adj_n = nm.adjacency_list(nodes, edges)
    for name, f in [("DFS (stack)", nm.dfs), ("BFS (queue)", nm.bfs)]:
        best = min(ev.measure_time(f, adj_n, nodes[0])[1] for _ in range(3))   # 3 回の最小値
        r = f(adj_n, nodes[0])
        times[name].append(best * 1e3)
        rows.append({"name": f"{name}, V={n}", "visited": r["visited"], "frontier": r["max_frontier"],
                     "ms": best * 1e3, "list": nm.memory_estimate(nodes, edges)["adjacency_list"]})
table(rows, [("name", "algorithm / size", "{}"), ("visited", "visited", "{}"),
             ("frontier", "max stack/queue length", "{}"), ("ms", "runtime (ms, measured)", "{:.2f}"),
             ("list", "adjacency list size", "{}")])
show(cc.plot_loglog(sizes, times, "DFS vs BFS runtime on ring networks (measured on this machine)", "runtime (ms)",
                    colors={"DFS (stack)": cc.ALGO_COLORS["dfs"], "BFS (queue)": cc.ALGO_COLORS["bfs"]}),
     "04_dfs_bfs_runtime")

## 4. 協調関係のネットワーク図とネットワーク構造別の協調率

エッジの色 = ある 1 回のシミュレーションの最終年の関係状態。棒グラフは 100 回のモンテカルロ平均です。

In [ ]:
#@title 協調関係のネットワーク図と構造別の協調率
states = []
for k, e in NETS.items():
    out = sim.simulate(COUNTRIES, e, PARAMS, M, years=YEARS, run=0)
    states.append(out["records"][-1]["states"])
show(cc.plot_networks([(f"{k.capitalize()} (year {YEARS}, one run)", IDS, e) for k, e in NETS.items()],
                      "Relationship states on each network (model)", states=states), "04_coop_networks")
summ = {k: sim.monte_carlo(COUNTRIES, e, PARAMS, M, RUNS, years=YEARS)["summary"] for k, e in NETS.items()}
keys = ["cooperation_rate", "cooperation_state_rate", "tension_rate", "conflict_rate"]
show(cc.plot_grouped([k.capitalize() for k in NETS], keys, {key: [summ[k][key] for k in NETS] for key in keys},
                     f"Cooperation and conflict by network structure ({RUNS} runs, model)", "share",
                     colors={"cooperation_rate": "#3d3c39", "cooperation_state_rate": cc.STATE_COLORS["Cooperation"],
                             "tension_rate": cc.STATE_COLORS["Tension"], "conflict_rate": cc.STATE_COLORS["Conflict"]},
                     errors={key: [summ[k][key + "_sd"] for k in NETS] for key in keys}, ylim=(0, 1.0)),
     "04_by_structure")

**注意**：サンプルの 5 国は属性が違うので、ネットワーク構造を変えると「どの国どうしがつながるか」も変わります。構造そのものの効果を調べるには、全員同じ属性の国家で比べてください（Experiment 5）。

## 5. 学生向け TODO

In [ ]:
# TODO 1：ネットワーク構造を変更する / TODO 2：ノード数を変更する
N_NODES = 8
ids8 = nm.node_names(N_NODES)
same = [ag.make_country(i, 0.6, 0.5, 0.5, "conditional") for i in ids8]     # 全員同じ属性
for kind in nm.NETWORK_TYPES:
    e8 = nm.make_network(kind, ids8, 0.4, 7)
    s8 = sim.monte_carlo(same, e8, PARAMS, M, 50, years=YEARS)["summary"]
    print(f"{kind:7} E={len(e8):2}: Cooperation 状態 {s8['cooperation_state_rate']:.3f}  Conflict {s8['conflict_rate']:.3f}")
# TODO 3：DFS と BFS を比較する（star・line ネットワークで、スタック／キューの最大の長さはどう違うか）

## 考察（チームで記入）

- **TODO**：DFS と BFS は、到達できるノード・経路・記憶域・実行時間でどう違ったか
- **TODO**：ネットワーク構造によって、協調・紛争状態の割合は変わったか。その違いは構造によるものか、国家の属性によるものか（RQ4）
- **TODO**：隣からの波及（`spillover_effect`）を 0 にすると、構造の違いはどうなるか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 スタック・キュー、第5章 再帰、第6章 単純挿入ソート）
- R. Axelrod, *The Evolution of Cooperation*, Basic Books, 1984（繰り返し囚人のジレンマと Tit-for-Tat）
- G2 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_countries.json`、`data/payoff_matrix.json`、`data/simulation_parameters.json`（この教材のための**架空のデータ**）